[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/20_capstone/20_proyecto_capstone.ipynb)

# Proyecto 20 · Capstone: Políglota completo

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración** | ≈ 3.5 h |
| **GPU** | No necesaria en `FAST_DEV_RUN` (≈ 1 min de CPU); T4/L4 con modelos reales y `FAST_DEV_RUN=False` |
| **Requiere** | Todo el curso; la lección 20 es la solución de referencia del procedimiento |
| **Entrega** | `poliglota_informe.md` + `poliglota_config.json` + este notebook ejecutado |

## 🏢 Contexto de negocio
Es el día de la revisión de lanzamiento de **Políglota v1.0**. La CTO ya fijó los **criterios de aceptación** (celda dada). Tú decides cada componente con datos de *dev*, montas el servicio, lo evalúas **una vez** en el test ciego y entregas el informe técnico que se leerá en la reunión. La pregunta no es «¿cuánto da?», sino **«¿podemos lanzar, con qué riesgos y qué medimos después?»**.

## 📦 Datos
Corpus Políglota canónico (`carga_corpus`, módulo 00: Wikipedia ES/EN + CodeSearchNet; sin red, **sintético** + stdlib con aviso). Split **dev / test ciego** estratificado por idioma (dado). Embedders v1/v2 (e5 o fallbacks, módulo 18). Opcional: el retriever visual del proyecto 16 como tercer componente.

## 🎯 Entregables y rúbrica final (100 puntos)
| # | Entregable | Criterio objetivo (calibrado con la solución de referencia) | Pts |
|---|---|---|---|
| 1 | Informe del tokenizer (BPE byte-level, HF `tokenizers`) | fertilidad y bytes/token por idioma; el código paga más que el español | 10 |
| 2 | `elige_configuracion()` decidida **solo con dev** | config válida; nDCG@10 en dev ≥ 0.884 (ref. 0.914) | 25 |
| 3 | Índice verificado | neighbor recall@10 ≥ 0.95 con N = 20 000 y proyección de RAM a 10 M | 10 |
| 4 | Servicio montado con la config | p99 frío ≤ 50 ms (CPU, batch=1); hit rate Zipf ≥ 0.70; versión servida con `dim` | 15 |
| 5 | Evaluación en test ciego (una vez) | nDCG@10 del servicio ≥ 0.857 (ref. 0.887; BM25 0.842); IC pareado vs BM25 y slices | 20 |
| 6 | Informe técnico generado | secciones obligatorias + tabla de los 7 criterios + archivos guardados | 15 |
| 7 | Reflexión | 3 respuestas al final | 5 |

**Aprobado: ≥ 75/100.** Reglas de juego: (a) nada de lo que elijas puede mirar `I_TEST` (la rúbrica revisa el código de la solución, tú revisa el tuyo); (b) el número final sale **del servicio**, no de una matriz de scores.

## 0. Preparación (celdas dadas)

In [ ]:
!pip install -q faiss-cpu qdrant-client fastapi httpx sentence-transformers scikit-learn matplotlib pandas tokenizers

In [ ]:
import os, sys, time, math, random, json, hashlib, unicodedata, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True       # True: tamaños diminutos (CPU en minutos). False en Colab con GPU para modelos reales y más datos.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
try:
    import torch
    torch.manual_seed(SEED); torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch, device = None, "cpu"
AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
def bootstrap_ci(values, n_boot: int = 2000, alpha: float = 0.05, seed: int = 42):
    """IC percentil de la media remuestreando QUERIES (la unidad de muestreo es la query, no el par query-doc)."""
    v = np.asarray(values, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(v), size=(n_boot, len(v)))
    means = v[idx].mean(axis=1)
    return float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def paired_bootstrap(a, b, n_boot: int = 10000, alpha: float = 0.05, seed: int = 42) -> dict:
    """Test pareado A vs B sobre las MISMAS queries. IC percentil de la diferencia media y p-valor bilateral
    por 'shift' (se centra la diferencia en 0 bajo H0 y se cuenta cuántas veces el remuestreo supera lo observado)."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(n_boot, len(d)))
    means = d[idx].mean(axis=1)
    null = (d - d.mean())[idx].mean(axis=1)
    p = (np.sum(np.abs(null) >= abs(d.mean())) + 1) / (n_boot + 1)
    return {"diff": float(d.mean()), "ci_low": float(np.quantile(means, alpha / 2)),
            "ci_high": float(np.quantile(means, 1 - alpha / 2)), "p_value": float(p)}

def paired_permutation(a, b, n_perm: int = 10000, seed: int = 42) -> float:
    """Test de aleatorización pareado (sign-flip): bajo H0 el signo de cada diferencia por query es intercambiable."""
    d = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    signs = np.random.default_rng(seed).choice([-1.0, 1.0], size=(n_perm, len(d)))
    stat = np.abs((signs * d).mean(axis=1))
    return float((np.sum(stat >= abs(d.mean()) - 1e-12) + 1) / (n_perm + 1))

def holm(pvalues: dict, alpha: float = 0.05) -> dict:
    """Corrección de Holm-Bonferroni para comparaciones múltiples. Devuelve {nombre: (p_ajustado, rechaza_H0)}."""
    items = sorted(pvalues.items(), key=lambda kv: kv[1])
    m, out, running = len(items), {}, 0.0
    for i, (name, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        out[name] = (running, running < alpha)
    return out

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels = corpus["docs"], corpus["queries"], corpus["qrels"]
doc_ids = [d["id"] for d in docs]; doc_texts = [d["text"] for d in docs]
doc_lang = np.array([d["lang"] for d in docs])
qids = [q["qid"] for q in queries]; q_texts = [q["text"] for q in queries]; q_lang = np.array([q["lang"] for q in queries])
print(f"fuente={corpus['source']} · {len(docs)} docs · {len(queries)} queries · idiomas={sorted(set(doc_lang))}")

In [ ]:
def run_desde_scores(S, k: int = 10, ids=None) -> dict:
    """Matriz de scores (n_queries x n_docs) -> run {qid: [doc_ids top-k]} con el desempate canónico."""
    ids = doc_ids if ids is None else ids
    return {qid: rank_from_scores(S[i], ids, k) for i, qid in enumerate(qids)}

def ndcg_por_query(run: dict, k: int = 10) -> np.ndarray:
    """nDCG@k de cada query (orden de `qids`); unidad de remuestreo para bootstrap."""
    return np.array([ndcg_at_k(run.get(q, []), qrels[q], k) for q in qids])

def l2n(X):
    X = np.asarray(X, dtype=np.float32)
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)

BM25_IDX = BM25().fit([tokenize_simple(t) for t in doc_texts], doc_ids)
S_BM25 = np.array([BM25_IDX.scores(tokenize_simple(q)) for q in q_texts])
print("BM25 (baseline del módulo 00):", {m: round(v, 3) for m, v in evaluate(run_desde_scores(S_BM25), qrels).items() if m in ("ndcg", "recall", "mrr")})

In [ ]:
# Dos versiones del embedder de Políglota: v1 (la que está en producción) y v2 (la candidata).
# Con hub: v1 = multilingual-e5-small (384 d), v2 = multilingual-e5-base (768 d).
# Sin hub: dos fallbacks TF-IDF+SVD DISTINTOS (otro analizador y otra dimensión) -> espacios incompatibles, como en la vida real.
USAR_MODELOS_REALES = True
V1_REAL, V2_REAL = "multilingual-e5-small", "multilingual-e5-base"

def carga_versiones(textos_ajuste) -> dict:
    vers = {}
    if USAR_MODELOS_REALES:
        for nombre, clave in (("v1", V1_REAL), ("v2", V2_REAL)):
            e = cargar_embedder(clave)          # None si no hay hub
            if e is None:
                break
            vers[nombre] = e
    if len(vers) < 2:
        print("⚠️  Sin modelos reales: v1/v2 = fallbacks TF-IDF+SVD (NO son modelos neuronales; las cifras no representan a e5).")
        vers = {"v1": Embedder("v1 · char 3-4 + SVD 48d (fallback)", {}, fallback=HashSVDEmbedder(dim=48, ngram=(3, 4)).fit(textos_ajuste)),
                "v2": Embedder("v2 · char 2-5 + SVD 128d (fallback)", {}, fallback=HashSVDEmbedder(dim=128, ngram=(2, 5)).fit(textos_ajuste))}
    return vers

VERS = carga_versiones(doc_texts)
EMB = {}
for v, e in VERS.items():
    t0 = time.perf_counter()
    D, Qv = e.encode_docs(doc_texts), e.encode_queries(q_texts)
    EMB[v] = {"D": D, "Q": Qv, "seg": time.perf_counter() - t0}
    print(f"{v}: {e.clave} · dim={D.shape[1]} · nDCG@10={evaluate(run_desde_scores(Qv @ D.T), qrels)['ndcg']:.3f} · {EMB[v]['seg']:.1f}s")

In [ ]:
from collections import OrderedDict

def normaliza_query(texto: str) -> str:
    """Normalización ANTES de la clave de caché: NFC + minúsculas + espacios colapsados (sube el hit rate)."""
    return " ".join(unicodedata.normalize("NFC", texto).lower().split())

def clave_cache(version: str, texto: str) -> str:
    """La versión del modelo VA EN LA CLAVE: si no, tras migrar servirías vectores v1 contra un índice v2."""
    return hashlib.sha1(f"{version}\x00{normaliza_query(texto)}".encode("utf-8")).hexdigest()

class LRUCache:
    """Caché LRU O(1) con OrderedDict y contadores de aciertos/fallos."""
    def __init__(self, capacidad: int):
        self.cap, self.d, self.hits, self.misses = capacidad, OrderedDict(), 0, 0
    def get(self, k):
        if k in self.d:
            self.d.move_to_end(k); self.hits += 1
            return self.d[k]
        self.misses += 1
        return None
    def put(self, k, v):
        self.d[k] = v; self.d.move_to_end(k)
        if len(self.d) > self.cap:
            self.d.popitem(last=False)          # expulsa el menos usado recientemente
    @property
    def hit_rate(self) -> float:
        n = self.hits + self.misses
        return self.hits / n if n else 0.0

In [ ]:
import faiss
faiss.omp_set_num_threads(1)

class ServicioPoliglota:
    """Búsqueda versionada: caché de embeddings de query + índice denso por versión (FAISS IndexFlatIP) + BM25,
    filtro por idioma, fusión RRF, alias 'prod' -> versión y versiones 'adaptador' (queries nuevas -> índice viejo)."""
    def __init__(self, docs, bm25, cache_cap: int = 2048, rrf_k: int = 60, profundidad: int = 50, tokenizador=None,
                 fusion: str = "rrf", alfa: float = 0.5):
        # CONTRATO: `tokenizador` debe ser el MISMO con el que se construyó `bm25` (si no, fallo silencioso F3 del módulo 19)
        self.tok = tokenizador or tokenize_simple
        self.fusion, self.alfa = fusion, alfa      # fusion: 'rrf' | 'lineal' (alfa·denso + (1-alfa)·BM25, min-max en cada lista)
        self.ids = [d["id"] for d in docs]; self.lang = np.array([d["lang"] for d in docs])
        self.bm25, self.cache, self.rrf_k, self.prof = bm25, LRUCache(cache_cap), rrf_k, profundidad
        self.versiones, self.alias, self.lat_ms = {}, {}, []

    def registrar_version(self, nombre, embedder, D, indice: str = "flat", ef_search: int = 64):
        """indice: 'flat' (exacto) | 'hnsw' (FAISS IndexHNSWFlat, M=32)."""
        if indice == "hnsw":
            idx = faiss.IndexHNSWFlat(D.shape[1], 32, faiss.METRIC_INNER_PRODUCT); idx.hnsw.efConstruction = 80
        else:
            idx = faiss.IndexFlatIP(D.shape[1])
        idx.add(l2n(D))
        self.versiones[nombre] = {"emb": embedder, "idx": idx, "W": None, "base": nombre, "indice": indice, "ef": ef_search}

    def registrar_adaptador(self, nombre, embedder, W, sobre: str):
        """Queries con `embedder` (versión nueva) proyectadas con W al espacio del índice `sobre` (versión vieja)."""
        base = self.versiones[sobre]
        self.versiones[nombre] = {"emb": embedder, "idx": base["idx"], "W": W, "base": sobre, "indice": base["indice"], "ef": base["ef"]}

    def embebe_query(self, texto, nombre):
        k = clave_cache(nombre, texto)
        v = self.cache.get(k)
        if v is None:
            cfg = self.versiones[nombre]
            v = cfg["emb"].encode_queries([texto])
            if cfg["W"] is not None:
                v = l2n(v @ cfg["W"])
            self.cache.put(k, v)
        return v

    def _rangos_densos(self, texto, nombre, cand):
        return [int(i) for i in self._cands_densos(texto, nombre, cand)[0]]

    def _cands_densos(self, texto, nombre, cand):
        """(índices, scores) del top-`profundidad` denso, con el filtro aplicado DENTRO de la búsqueda."""
        cfg = self.versiones[nombre]; idx = cfg["idx"]; q = self.embebe_query(texto, nombre)
        sel = faiss.IDSelectorBatch(cand.astype("int64")) if cand is not None else None
        if cfg["indice"] == "hnsw":       # HNSW exige sus propios parámetros (efSearch + selector de filtro)
            params = faiss.SearchParametersHNSW(efSearch=cfg["ef"]) if sel is None else faiss.SearchParametersHNSW(sel=sel, efSearch=cfg["ef"])
        else:
            params = faiss.SearchParameters(sel=sel) if sel is not None else None
        Dsc, I = idx.search(q, min(self.prof, idx.ntotal), params=params)
        m = I[0] >= 0
        return I[0][m], Dsc[0][m]

    def _lineal(self, texto, nombre, cand):
        """Combinación convexa con min-max dentro de cada lista (un doc ausente de una lista aporta 0 por ella)."""
        I, sd = self._cands_densos(texto, nombre, cand)
        sb_all = self.bm25.scores(self.tok(texto))
        Ib = np.array(self._rangos_bm25(texto, cand), dtype=int); sb = sb_all[Ib] if len(Ib) else np.zeros(0)
        def mm(v):
            return (v - v.min()) / max(float(v.max() - v.min()), 1e-9) if len(v) else v
        score = {}
        for i, v in zip(I, mm(sd)):
            score[int(i)] = score.get(int(i), 0.0) + self.alfa * float(v)
        for i, v in zip(Ib, mm(sb)):
            score[int(i)] = score.get(int(i), 0.0) + (1 - self.alfa) * float(v)
        return score

    def _rangos_bm25(self, texto, cand):
        s = self.bm25.scores(self.tok(texto))
        if cand is not None:
            m = np.full(len(s), -np.inf); m[cand] = s[cand]; s = m
        orden = np.lexsort((np.arange(len(s)), -s))[: self.prof]
        return [int(i) for i in orden if np.isfinite(s[i]) and s[i] > 0]

    def buscar(self, texto, k: int = 10, lang=None, modo: str = "hibrido", version: str = "prod"):
        t0 = time.perf_counter()
        nombre = self.alias.get(version, version)
        cand = np.flatnonzero(self.lang == lang) if lang else None
        if modo == "hibrido" and self.fusion == "lineal":
            score = self._lineal(texto, nombre, cand)
        else:
            listas = []
            if modo in ("denso", "hibrido"):
                listas.append(self._rangos_densos(texto, nombre, cand))
            if modo in ("bm25", "hibrido"):
                listas.append(self._rangos_bm25(texto, cand))
            score = {}
            for lst in listas:                       # RRF: sum 1/(k + rango), rango desde 1
                for r, i in enumerate(lst):
                    score[i] = score.get(i, 0.0) + 1.0 / (self.rrf_k + r + 1)
        top = sorted(score, key=lambda i: (-score[i], i))[:k]
        self.lat_ms.append((time.perf_counter() - t0) * 1e3)
        return [(self.ids[i], round(score[i], 5)) for i in top]

    def metricas(self) -> dict:
        lat = np.array(self.lat_ms) if self.lat_ms else np.zeros(1)
        return {"n": len(self.lat_ms), "p50_ms": float(np.percentile(lat, 50)), "p99_ms": float(np.percentile(lat, 99)),
                "hit_rate": self.cache.hit_rate, "versiones": sorted(self.versiones), "alias": dict(self.alias)}

In [ ]:
import re
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def split_estratificado(queries, frac_test: float = 0.5, seed: int = SEED):
    """dev / test CIEGO estratificado por idioma (módulo 15). Todo lo que se elige, se elige en dev."""
    rng = random.Random(seed); dev, test = [], []
    for lang in sorted({q["lang"] for q in queries}):
        grp = [q for q in queries if q["lang"] == lang]; rng.shuffle(grp)
        cut = int(round(len(grp) * frac_test)); test += grp[:cut]; dev += grp[cut:]
    return dev, test

DEV, TEST = split_estratificado(queries)
I_DEV = np.array([qids.index(q["qid"]) for q in DEV]); I_TEST = np.array([qids.index(q["qid"]) for q in TEST])
print(f"dev={len(DEV)} · test ciego={len(TEST)} queries (estratificado por idioma)")

def entrena_bpe(textos, vocab=4000):
    """BPE byte-level con HF tokenizers (módulos 02-03): NFC + ByteLevel + BPE."""
    tk = Tokenizer(models.BPE(unk_token=None)); tk.normalizer = normalizers.NFC()
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True); tk.decoder = decoders.ByteLevel()
    tk.train_from_iterator(textos, trainers.BpeTrainer(vocab_size=vocab, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
    return tk

def tok_ident(texto: str) -> list[str]:
    """Analizador léxico de Políglota: palabras + partes de identificadores (snake_case / camelCase), módulos 00 y 13."""
    out = tokenize_simple(texto)
    for w in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", texto):
        partes = re.findall(r"[A-Z]?[a-z]+|[A-Z]+(?![a-z])|\d+", w.replace("_", " "))
        if len(partes) > 1:
            out += [p.lower() for p in partes]
    return out

class EmbedderRecortado:
    """Toma las primeras `dim` coordenadas y renormaliza (Matryoshka; con el fallback SVD equivale a PCA truncada)."""
    def __init__(self, base, dim):
        self.base, self.dim, self.clave = base, dim, f"{base.clave} [:{dim}]"
    def encode_queries(self, t):
        return l2n(self.base.encode_queries(t)[:, :self.dim])
    def encode_docs(self, t):
        return l2n(self.base.encode_docs(t)[:, :self.dim])

def rrf_mat(*Ss, k=60):
    out = np.zeros_like(Ss[0], dtype=float)
    for S in Ss:
        out += 1.0 / (k + np.argsort(np.argsort(-S, axis=1, kind="stable"), axis=1, kind="stable") + 1)
    return out

def minmax(S):
    lo, hi = S.min(1, keepdims=True), S.max(1, keepdims=True)
    return (S - lo) / np.maximum(hi - lo, 1e-9)

def lineal_lista(S1, S2, a, prof=50):
    """Igual que el servicio: cada lista se corta a su top-`prof`, se normaliza min-max dentro de la lista y los docs
    fuera de una lista aportan 0 por ella. S1 = denso, S2 = BM25 (n_queries x n_docs)."""
    def norm(S):
        thr = -np.sort(-S, axis=1)[:, min(prof, S.shape[1]) - 1][:, None]
        Sc = np.maximum(S, thr)
        return (Sc - thr) / np.maximum(Sc.max(1, keepdims=True) - thr, 1e-9)
    return a * norm(S1) + (1 - a) * norm(S2)

def int8_ida_vuelta(D):
    """Cuantiza a int8 (calibrado por dimensión) y decodifica: simula la pérdida de precisión del índice."""
    lo, hi = np.quantile(D, 0.001, axis=0), np.quantile(D, 0.999, axis=0); esc = np.maximum(hi - lo, 1e-9) / 255
    return (np.clip(np.round((D - lo) / esc) - 128, -128, 127) + 128) * esc + lo

pq_S = lambda S: ndcg_por_query(run_desde_scores(S))      # nDCG@10 por query (orden de qids)

In [ ]:
def corre(f):
    """Ejecuta un chequeo sin romper 'ejecutar todo' si el TODO está pendiente."""
    try:
        r = f()
        print(r if r is not None else "✅ OK")
        return True
    except NotImplementedError as e:
        print(f"⏳ pendiente: {e}")
    except AssertionError as e:
        print(f"❌ falló: {e}")
    except Exception as e:  # noqa: BLE001
        print(f"❌ {type(e).__name__}: {e}")
    return False

REF = {'dev_config': 0.914, 'test': 0.887, 'test_bm25': 0.842}

In [ ]:
ACEPTACION = {
    "calidad_vs_bm25":   "nDCG@10 en test ≥ BM25 del módulo 00 y límite inferior del IC95 de la diferencia pareada > −0.03",
    "peor_slice":        "ningún idioma más de 0.05 por debajo de BM25 en test",
    "recall_ann":        "neighbor recall@10 del índice elegido ≥ 0.95 frente a fuerza bruta",
    "latencia_p99_ms":   "p99 del servicio ≤ 50 ms (CPU, batch=1)",
    "ram_10M_gb":        "RAM del índice para 10 M de chunks ≤ 16 GB por réplica",
    "coste_reembed_usd": "re-embedding de 10 M chunks × 400 tokens ≤ 500 $ (API a 0.02 $/1M, oct-2026)",
    "hit_rate":          "hit rate de la caché con tráfico Zipf ≥ 0.70",
}

In [ ]:
ANALIZADORES = {"palabras": tokenize_simple, "palabras+ident": tok_ident}
def trafico_zipf(n, universo, s=1.1, p_variante=0.35, seed=SEED):
    """(Dado) tráfico Zipf con variantes superficiales (módulo 18)."""
    rng = np.random.default_rng(seed); p = 1.0 / np.arange(1, len(universo) + 1) ** s; p /= p.sum(); out = []
    for i in rng.choice(len(universo), size=n, p=p):
        t = universo[i]
        out.append(rng.choice([t.upper(), "  " + t, t + " "]) if rng.random() < p_variante else t)
    return out
UNIVERSO = [q_texts[i % len(q_texts)] + ("" if i < len(q_texts) else f" {i // len(q_texts)}") for i in range(3000)]

def vectores_de(config):
    """(Dado) Vectores de docs y queries según config: modelo, dim (primeras d + renormalizar) y precisión."""
    D, Q = EMB[config["modelo"]]["D"][:, :config["dim"]], EMB[config["modelo"]]["Q"][:, :config["dim"]]
    D, Q = l2n(D), l2n(Q)
    return (int8_ida_vuelta(D) if config["precision"] == "int8" else D), Q

def scores_de(config):
    """(Dado) Matriz de scores (queries x docs) del sistema descrito por config, SIN índice ANN (fuerza bruta)."""
    D, Q = vectores_de(config); f = ANALIZADORES[config["analizador"]]
    bm = BM25().fit([f(t) for t in doc_texts], doc_ids); Sl = np.array([bm.scores(f(q)) for q in q_texts]); Sd = Q @ D.T
    if config["fusion"] == "rrf":
        return rrf_mat(Sd, Sl)
    if config["fusion"] == "lineal":
        return lineal_lista(Sd, Sl, config["alfa"])
    return Sd if config["fusion"] == "denso" else Sl
RES = {}

In [ ]:
CLAVES_CONFIG = {"analizador", "modelo", "dim", "precision", "fusion", "alfa", "ef_search"}

def _requiere(*claves):
    falta = [k for k in claves if k not in RES]
    if falta:
        raise NotImplementedError(f"completa antes los TODOs que producen {falta}")

def test_tokenizer():
    df = informe_tokenizer(4000); RES["tok"] = df
    assert set(df["lang"]) == {"es", "en", "code"} and {"fertilidad", "bytes_por_token"} <= set(df.columns)
    f = dict(zip(df["lang"], df["fertilidad"]))
    assert f["code"] > f["es"], "el código debería pagar más tokens por palabra que el español"
    return f"✅ tokenizer · fertilidad {', '.join(f'{k}={v:.2f}' for k, v in f.items())}"

def test_config():
    cfg = elige_configuracion(); RES["config"] = cfg
    assert CLAVES_CONFIG <= set(cfg), f"faltan claves: {CLAVES_CONFIG - set(cfg)}"
    assert cfg["analizador"] in ANALIZADORES and cfg["modelo"] in EMB and cfg["precision"] in ("float32", "int8")
    assert cfg["fusion"] in ("rrf", "lineal", "denso", "bm25") and 1 <= cfg["dim"] <= EMB[cfg["modelo"]]["D"].shape[1]
    dev = pq_S(scores_de(cfg))[I_DEV].mean(); RES["dev_config"] = dev
    assert dev >= REF["dev_config"] - 0.03, f"nDCG@10 en dev de tu configuración {dev:.3f} < {REF['dev_config'] - 0.03:.3f}"
    return f"✅ configuración · dev nDCG@10 = {dev:.3f} · {cfg}"

def test_indice():
    _requiere("config")
    r = verifica_indice(RES["config"]); RES.update(recall_ann=r["recall"], ram_10M_gb=r["ram_10M_gb"])
    assert r["recall"] >= 0.95, f"recall@10 {r['recall']:.3f} < 0.95: sube efSearch"
    return f"✅ índice · recall@10={r['recall']:.3f} · RAM 10 M={r['ram_10M_gb']:.1f} GB"

def test_servicio():
    _requiere("config")
    svc, m = monta_servicio(RES["config"]); RES["svc"] = svc; RES.update(m)
    assert m["p99_frio"] <= 50, f"p99 {m['p99_frio']:.1f} ms > 50 ms"
    assert m["hit_rate"] >= 0.70, f"hit rate {m['hit_rate']:.2f} < 0.70"
    assert svc.alias.get("prod") in svc.versiones and str(RES["config"]["dim"]) in svc.alias["prod"], "la versión servida debe incluir dim"
    return f"✅ servicio · p99 frío={m['p99_frio']:.1f} ms · hit rate={m['hit_rate']:.2f} · versión={svc.alias['prod']}"

def test_evaluacion():
    _requiere("config", "svc")
    e = evalua_test(RES["svc"]); RES["eval"] = e
    assert {"final", "bm25", "delta_bm25", "slices"} <= set(e)
    assert e["final"] >= REF["test"] - 0.03, f"nDCG@10 en test {e['final']:.3f} < {REF['test'] - 0.03:.3f}"
    return f"✅ evaluación · test nDCG@10={e['final']:.3f} (BM25 {e['bm25']:.3f}) · Δ IC95=[{e['delta_bm25']['ci_low']:+.3f},{e['delta_bm25']['ci_high']:+.3f}]"

def test_informe():
    _requiere("config", "eval", "tok", "recall_ann", "p99_frio", "dev_config")
    md = genera_informe(); RES["informe"] = md
    for h in ("Decisiones", "test ciego", "Costos", "Criterios de aceptación", "Limitaciones"):
        assert h.lower() in md.lower(), f"falta la sección «{h}»"
    assert md.count("✅") + md.count("❌") >= len(ACEPTACION), "la tabla de aceptación debe tener una fila por criterio"
    assert os.path.exists("poliglota_informe.md") and os.path.exists("poliglota_config.json")
    return f"✅ informe · {md.count('✅')} criterios cumplidos de {len(ACEPTACION)}"

def rubrica_final():
    filas = [("1 tokenizer", 10, test_tokenizer), ("2 decisiones en dev", 25, test_config), ("3 índice", 10, test_indice),
             ("4 servicio", 15, test_servicio), ("5 evaluación test", 20, test_evaluacion), ("6 informe", 15, test_informe)]
    total = 0
    for nombre, pts, f in filas:
        ok = corre(f); total += pts if ok else 0
        print(f"   {'✅' if ok else '⬜'} {nombre:22s} {pts:3d} pts")
    print(f"\nPuntuación automática: {total}/95 (+5 de la reflexión, la revisa una persona). Aprobado: ≥ 75.")
    if corpus["source"] != "synthetic":
        print("ℹ️ Datos reales: los umbrales de calidad están calibrados con el corpus sintético; compara con criterio.")
    return total

## 1. Tokenizer: informe de costo por idioma (TODO 1)
Entrena un BPE byte-level con `entrena_bpe(doc_texts, V)` y devuelve un `DataFrame` con columnas `lang`, `fertilidad` (tokens/palabra) y `bytes_por_token` para `es`, `en` y `code`.

<details><summary>🪜 Pista 1 (la idea)</summary>

Agrupa los documentos por `d['lang']`, codifica cada texto con `tk.encode(t).ids` y suma tokens, palabras (`t.split()`) y bytes (`t.encode()`).
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`n_tok = sum(len(tk.encode(t).ids) for t in tx)`; `fertilidad = n_tok / sum(len(t.split()) for t in tx)`; `bytes_por_token = sum(len(t.encode()) for t in tx) / n_tok`.
</details>

In [ ]:
def informe_tokenizer(V: int = 4000) -> pd.DataFrame:
    # TODO 1
    raise NotImplementedError("TODO 1: informe_tokenizer")

corre(test_tokenizer)

## 2. Decisiones en dev (TODO 2)
Devuelve la configuración completa decidida **solo con `I_DEV`**: `analizador` ∈ `ANALIZADORES`, `modelo` ∈ {`v1`,`v2`}, `dim`, `precision`, `fusion` ∈ {`rrf`,`lineal`,`denso`,`bm25`}, `alfa` y `ef_search`. Usa (o mejora, justificándolo) las reglas de la lección: modelo por test pareado, menor `dim` a ≤ 0.01 de la completa, int8 si pierde ≤ 0.01, lineal solo si su CV gana a RRF con IC.

<details><summary>🪜 Pista 1 (la idea)</summary>

`scores_de(config)` te da la matriz de scores de cualquier configuración: evalúala con `pq_S(S)[I_DEV]`. Decide por etapas (analizador → modelo → dim → precisión → fusión) en vez de una rejilla enorme (¿recuerdas el *winner's curse*?).
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

Analizador: `max(ANALIZADORES, key=lambda a: pq_S(scores_de({**cfg, 'analizador': a, 'fusion': 'bm25'}))[I_DEV].mean())`. Fusión lineal con CV: la celda §4 de la lección (`PQa`, `np.array_split`, `paired_bootstrap(lin_cv, pq_rrf)`). `ef_search`: empieza en 128 y verifícalo en el TODO 3.
</details>

In [ ]:
def elige_configuracion() -> dict:
    # TODO 2: SOLO I_DEV. Devuelve dict(analizador, modelo, dim, precision, fusion, alfa, ef_search)
    raise NotImplementedError("TODO 2: elige_configuracion")

corre(test_config)

## 3. Índice: recall y memoria (TODO 3)
Amplía los vectores de documentos de tu config con copias ruidosas hasta N = 20 000 (como en la lección §5), mide el neighbor recall@10 de HNSW (M = 32, `efSearch = config['ef_search']`) contra Flat con las queries de dev, y proyecta la RAM a 10 M de chunks (bytes por vector + 256 B de grafo).

<details><summary>🪜 Pista 1 (la idea)</summary>

Usa `vectores_de(config)`; ruido gaussiano de escala `0.35/sqrt(dim)` y renormaliza.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`flat = faiss.IndexFlatIP(d); flat.add(X); gt = flat.search(Qd, 10)[1]`; `hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)`; `hnsw.search(Qd, 10, params=faiss.SearchParametersHNSW(efSearch=ef))[1]`; RAM = `1e7 * (bpv + 256) / 1e9`.
</details>

In [ ]:
def verifica_indice(config: dict, N: int = 20_000) -> dict:
    """Devuelve {"recall": float, "ram_10M_gb": float}."""
    # TODO 3
    raise NotImplementedError("TODO 3: verifica_indice")

corre(test_indice)

## 4. Servicio (TODO 4)
Monta `ServicioPoliglota` con tu config (BM25 con **el mismo analizador** que pasas como `tokenizador`, `EmbedderRecortado`, vectores de `vectores_de`, `indice='hnsw'`, tu `ef_search` y tu fusión), con alias `prod` → una versión cuyo nombre incluya modelo, `dim` y precisión. Mide p50/p99 en frío (todas las queries una vez) y el hit rate con `trafico_zipf(5000, UNIVERSO)` en un servicio nuevo.

<details><summary>🪜 Pista 1 (la idea)</summary>

La lección §6 hace exactamente esto. Recuerda: la fusión del servicio se pasa en el constructor (`fusion='rrf'|'lineal'`, `alfa`). Si tu config elige `denso` o `bm25`, usa `modo=` al buscar.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`svc = ServicioPoliglota(docs, bm, cache_cap=1024, tokenizador=f, fusion=..., alfa=...)`; `svc.registrar_version(v, EmbedderRecortado(VERS[m], dim), D, indice='hnsw', ef_search=ef)`; `svc.alias['prod'] = v`.
</details>

In [ ]:
def monta_servicio(config: dict):
    """Devuelve (servicio, {"p50_frio", "p99_frio", "hit_rate"}). El servicio devuelto es el que se evalúa en test."""
    # TODO 4
    raise NotImplementedError("TODO 4: monta_servicio")

corre(test_servicio)

## 5. Evaluación en test ciego, una sola vez (TODO 5)
Sirve las queries de **test** por el servicio y compáralas con BM25 del módulo 00 (`S_BM25`): nDCG@10 medio, `paired_bootstrap` y nDCG por idioma de ambos.

<details><summary>🪜 Pista 1 (la idea)</summary>

Construye un `run` solo para las queries de `I_TEST` llamando a `svc.buscar(q_texts[i], k=10, modo=...)` y evalúalo con `ndcg_at_k` contra `qrels`.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

`pq_f = np.array([ndcg_at_k([d for d, _ in svc.buscar(q_texts[i], k=10, modo=modo)], qrels[qids[i]], 10) for i in I_TEST])`; `pq_b = pq_S(S_BM25)[I_TEST]`; `slices = {l: (pq_f[q_lang[I_TEST]==l].mean(), pq_b[q_lang[I_TEST]==l].mean()) for l in ('es','en','code')}`.
</details>

In [ ]:
def evalua_test(svc) -> dict:
    """Devuelve {"final", "bm25", "delta_bm25" (paired_bootstrap), "slices": {lang: (final, bm25)}}."""
    # TODO 5
    raise NotImplementedError("TODO 5: evalua_test")

corre(test_evaluacion)

## 6. Informe técnico (TODO 6)
Genera el Markdown **desde `RES`** (nada de números a mano) con las secciones: *Decisiones*, *Resultados en test ciego*, *Costos y latencia*, *Criterios de aceptación* (tabla con ✅/❌ por cada criterio de `ACEPTACION`) y *Limitaciones*. Guárdalo en `poliglota_informe.md` y la config en `poliglota_config.json`.

<details><summary>🪜 Pista 1 (la idea)</summary>

Calcula primero un `dict` `checks` criterio → bool con los mismos umbrales del texto de `ACEPTACION`, luego formatea la tabla con un `join`.
</details>

<details><summary>🪜 Pista 2 (casi la API)</summary>

Costo: `1e7 * 400 / 1e6 * 0.02`; `checks['calidad_vs_bm25'] = e['final'] >= e['bm25'] and e['delta_bm25']['ci_low'] > -0.03`; `open('poliglota_informe.md','w').write(md)`; `json.dump(RES['config'], open('poliglota_config.json','w'))`.
</details>

In [ ]:
def genera_informe() -> str:
    # TODO 6
    raise NotImplementedError("TODO 6: genera_informe")

corre(test_informe)

In [ ]:
puntos = rubrica_final()

---
## ⛔ SPOILER — intenta resolverlo primero
Solución de referencia: el procedimiento de la lección 20 empaquetado en funciones. Ejecutarla redefine tus funciones y vuelve a pasar la rúbrica final.

In [ ]:
def informe_tokenizer(V: int = 4000) -> pd.DataFrame:
    tk = entrena_bpe(doc_texts, V); filas = []
    for l in ("es", "en", "code"):
        tx = [d["text"] for d in docs if d["lang"] == l]; n_tok = sum(len(tk.encode(t).ids) for t in tx)
        filas.append({"lang": l, "fertilidad": n_tok / sum(len(t.split()) for t in tx), "bytes_por_token": sum(len(t.encode()) for t in tx) / n_tok})
    return pd.DataFrame(filas)

def elige_configuracion() -> dict:
    dv = lambda cfg: pq_S(scores_de(cfg))[I_DEV]
    cfg = dict(analizador="palabras", modelo="v2", dim=EMB["v2"]["D"].shape[1], precision="float32", fusion="bm25", alfa=0.5, ef_search=128)
    cfg["analizador"] = max(ANALIZADORES, key=lambda a: dv({**cfg, "analizador": a}).mean())
    r = paired_bootstrap(dv({**cfg, "fusion": "denso", "modelo": "v2", "dim": EMB["v2"]["D"].shape[1]}),
                         dv({**cfg, "fusion": "denso", "modelo": "v1", "dim": EMB["v1"]["D"].shape[1]}))
    cfg["modelo"] = "v2" if r["ci_low"] > 0 else "v1"
    full = EMB[cfg["modelo"]]["D"].shape[1]
    dims = [d for d in (16, 32, 48, 64, 96, 128, 256, 384, 512, 768, 1024) if d <= full]
    nd_d = {d: dv({**cfg, "fusion": "denso", "dim": d}).mean() for d in dims}
    cfg["dim"] = min(d for d in dims if nd_d[d] >= nd_d[full] - 0.01)
    cfg["precision"] = "int8" if dv({**cfg, "fusion": "denso", "precision": "int8"}).mean() >= nd_d[cfg["dim"]] - 0.01 else "float32"
    alfas = np.linspace(0, 1, 21); PQa = np.array([dv({**cfg, "fusion": "lineal", "alfa": a}) for a in alfas])
    lin_cv = np.zeros(len(I_DEV))
    for f in np.array_split(np.random.default_rng(SEED).permutation(len(I_DEV)), 5):
        tr = np.setdiff1d(np.arange(len(I_DEV)), f); lin_cv[f] = PQa[PQa[:, tr].mean(1).argmax(), f]
    r = paired_bootstrap(lin_cv, dv({**cfg, "fusion": "rrf"}))
    cfg["fusion"] = "lineal" if r["ci_low"] > 0 else "rrf"; cfg["alfa"] = float(alfas[PQa.mean(1).argmax()])
    return cfg

def verifica_indice(config: dict, N: int = 20_000) -> dict:
    D, Q = vectores_de(config); d = D.shape[1]; rng = np.random.default_rng(SEED)
    base = D[rng.integers(0, len(D), N)]; X = l2n(base + 0.35 * rng.normal(size=base.shape).astype(np.float32) / np.sqrt(d))
    Qd = Q[I_DEV]; flat = faiss.IndexFlatIP(d); flat.add(X); gt = flat.search(Qd, 10)[1]
    h = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT); h.hnsw.efConstruction = 80; h.add(X)
    got = h.search(Qd, 10, params=faiss.SearchParametersHNSW(efSearch=config["ef_search"]))[1]
    bpv = d * (1 if config["precision"] == "int8" else 4)
    return {"recall": float(np.mean([len(set(a) & set(b)) / 10 for a, b in zip(gt, got)])), "ram_10M_gb": 1e7 * (bpv + 256) / 1e9}

def _servicio(config):
    f = ANALIZADORES[config["analizador"]]; bm = BM25().fit([f(t) for t in doc_texts], doc_ids)
    fus = config["fusion"] if config["fusion"] in ("rrf", "lineal") else "rrf"
    s = ServicioPoliglota(docs, bm, cache_cap=1024, tokenizador=f, fusion=fus, alfa=config["alfa"])
    D, _ = vectores_de(config); v = f"{config['modelo']}-d{config['dim']}-{config['precision']}"
    s.registrar_version(v, EmbedderRecortado(VERS[config["modelo"]], config["dim"]), D, indice="hnsw", ef_search=config["ef_search"])
    s.alias["prod"] = v
    return s

MODO = lambda config: "hibrido" if config["fusion"] in ("rrf", "lineal") else config["fusion"]

def monta_servicio(config: dict):
    s = _servicio(config)
    for t in q_texts: s.buscar(t, k=10, modo=MODO(config))
    lat = np.array(s.lat_ms)
    z = _servicio(config)
    for t in trafico_zipf(5000, UNIVERSO): z.buscar(t, k=10, modo=MODO(config))
    s.cache = LRUCache(1024); s.lat_ms = []           # el servicio a evaluar arranca limpio
    return s, {"p50_frio": float(np.percentile(lat, 50)), "p99_frio": float(np.percentile(lat, 99)), "hit_rate": z.cache.hit_rate}

def evalua_test(svc) -> dict:
    modo = MODO(RES["config"])
    pq_f = np.array([ndcg_at_k([d for d, _ in svc.buscar(q_texts[i], k=10, modo=modo)], qrels[qids[i]], 10) for i in I_TEST])
    pq_b = pq_S(S_BM25)[I_TEST]; lt = q_lang[I_TEST]
    return {"final": float(pq_f.mean()), "bm25": float(pq_b.mean()), "delta_bm25": paired_bootstrap(pq_f, pq_b),
            "slices": {l: (float(pq_f[lt == l].mean()), float(pq_b[lt == l].mean())) for l in ("es", "en", "code")}, "pq": pq_f, "pq_bm25": pq_b}

def genera_informe() -> str:
    c, e = RES["config"], RES["eval"]; coste = 1e7 * 400 / 1e6 * 0.02
    checks = {"calidad_vs_bm25": e["final"] >= e["bm25"] and e["delta_bm25"]["ci_low"] > -0.03,
              "peor_slice": all(f >= b - 0.05 for f, b in e["slices"].values()), "recall_ann": RES["recall_ann"] >= 0.95,
              "latencia_p99_ms": RES["p99_frio"] <= 50, "ram_10M_gb": RES["ram_10M_gb"] <= 16, "coste_reembed_usd": coste <= 500,
              "hit_rate": RES["hit_rate"] >= 0.70}
    filas = "\n".join(f"| {k} | {ACEPTACION[k]} | {'✅' if ok else '❌'} |" for k, ok in checks.items())
    tok = RES["tok"].set_index("lang")
    md = f"""# Políglota v1.0 · Informe técnico (proyecto)
Fuente: {corpus['source']} · dev {len(I_DEV)} / test ciego {len(I_TEST)} queries · semilla {SEED}

## Decisiones (solo dev)
`{json.dumps(c, ensure_ascii=False)}` · nDCG@10 en dev de la configuración: {RES['dev_config']:.3f}.
Fertilidad BPE 4k: es {tok.loc['es', 'fertilidad']:.2f} · en {tok.loc['en', 'fertilidad']:.2f} · code {tok.loc['code', 'fertilidad']:.2f} tokens/palabra.

## Resultados en test ciego
Servicio {e['final']:.3f} vs BM25 {e['bm25']:.3f} · Δ = {e['delta_bm25']['diff']:+.3f} (IC95 [{e['delta_bm25']['ci_low']:+.3f}, {e['delta_bm25']['ci_high']:+.3f}]) ·
por idioma (servicio/BM25): {', '.join(f'{l} {f:.3f}/{b:.3f}' for l, (f, b) in e['slices'].items())}.

## Costos y latencia
Re-embedding 10 M × 400 tokens: {coste:.0f} $ (API 0.02 $/1M, oct-2026) · RAM índice 10 M: {RES['ram_10M_gb']:.1f} GB · p50/p99 frío {RES['p50_frio']:.1f}/{RES['p99_frio']:.1f} ms (CPU) · hit rate {RES['hit_rate']:.2f} · recall ANN {RES['recall_ann']:.3f}.

## Criterios de aceptación
| Criterio | Umbral | OK |
|---|---|---|
{filas}

## Limitaciones
* {len(I_TEST)} queries de test: IC anchos; ampliar el golden set por slice con tráfico real.
* {'Corpus sintético y embedders fallback: repetir con datos y modelos reales.' if corpus['source'] == 'synthetic' else 'Validar online con interleaving antes del A/B.'}
* Latencia medida en CPU sin red: re-medir en el despliegue real.
"""
    open("poliglota_informe.md", "w", encoding="utf-8").write(md)
    json.dump({k: (v.item() if hasattr(v, "item") else v) for k, v in c.items()}, open("poliglota_config.json", "w"), ensure_ascii=False, indent=1)
    return md

In [ ]:
puntos = rubrica_final()
from IPython.display import Markdown, display
display(Markdown(RES["informe"]))

In [ ]:
# Figura final: nDCG@10 por idioma en test (servicio vs BM25) con IC95 global
e = RES["eval"]; fig, ax = plt.subplots(figsize=(8, 3.8)); x = np.arange(4); w = .38
for j, (nombre, pq, c) in enumerate((("BM25 (módulo 00)", e["pq_bm25"], GRIS), ("Políglota v1.0", e["pq"], AZUL))):
    lt = q_lang[I_TEST]; vals = [pq.mean()] + [pq[lt == l].mean() for l in ("es", "en", "code")]; ci = bootstrap_ci(pq)
    ax.bar(x + (j - .5) * w, vals, w, color=c, label=nombre)
    ax.errorbar(x[0] + (j - .5) * w, vals[0], yerr=[[vals[0] - ci[1]], [ci[2] - vals[0]]], color="k", capsize=3)
ax.set_xticks(x); ax.set_xticklabels(["global (IC95)", "es", "en", "code"]); ax.set_ylabel("nDCG@10 (test ciego)"); ax.set_ylim(0, 1.05)
ax.set_title("Políglota v1.0 en test ciego"); ax.legend(); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar:** el global con su IC y, sobre todo, el slice que más se mueve. Si el IC global incluye el 0, el informe debe decir «no peor que BM25, mejora no demostrada»: es una conclusión válida para lanzar una beta con interleaving, no para anunciar una mejora.

## 🚀 Retos extra
1. **Modelos reales**: con `FAST_DEV_RUN=False` en una T4, repite con `multilingual-e5-small/base` (o tu embedder del módulo 11) y compara las decisiones con las del fallback. ¿Cambia la fusión elegida?
2. **Reranker** (módulo 13): añade un cross-encoder sobre el top-20 y extiende `ACEPTACION` con su latencia.
3. **Multimodal** (módulo 16): añade el retriever visual del proyecto 16 como tercera lista en la RRF y un slice «capturas» al informe.
4. **Migración** (módulo 18): publica `v2-d…` junto a la versión actual, ajusta un adaptador y escribe en el informe el plan blue/green con costos.
5. **Interleaving** (módulo 18): simula clics con los qrels de test y compara tu servicio con BM25 por potencia.

## 🤔 Reflexión (5 pts)
1. ¿Qué decisión de tu configuración es la **más frágil** (la que más podría cambiar con datos reales) y qué medirías primero en la beta?
2. Si un criterio sale ❌, ¿lanzarías igualmente? Argumenta con el impacto y con el plan de mitigación.
3. ¿Qué parte del curso cambió más tu forma de **evaluar** un sistema de retrieval, y qué harías distinto en tu trabajo actual?